# 01 Original 10 Model Training

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`long_fit.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/06_final_validation/scripts/long_fit.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/06_final_validation)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'06_final_validation/scripts/long_fit.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '4baae91fb48e392ffe4ab4c1f3b85bfc61360e37'}}

## long_fit

원본: [06_final_validation/scripts/long_fit.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/06_final_validation/scripts/long_fit.py)  
Git blob: `4baae91fb48e392ffe4ab4c1f3b85bfc61360e37`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/06_final_validation)

In [ ]:
%%archive_source 06_final_validation/scripts/long_fit.py
"""Long-duration fit only; exact EXP_FINAL_001 recipes. Validation inputs never present."""
import os
for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS','BLIS_NUM_THREADS'): os.environ[k]='2'
import sys,time,json,hashlib,platform,gc,warnings,importlib.metadata
from pathlib import Path
import numpy as np,joblib,sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier,HistGradientBoostingClassifier
from sklearn.base import clone
from threadpoolctl import threadpool_limits,threadpool_info
BASE=Path(os.environ['BENCH_ROOT'])
ROOT=BASE/'sources'
for p in ('05_model_optimize/MLP/Normal/scripts','05_model_optimize/MLP/Combined/scripts','05_model_optimize/KNN/Normal/scripts','05_model_optimize/KNN/Combined/scripts','05_model_optimize/SVC/Normal/scripts','05_model_optimize/SVC/Combined/scripts','05_model_optimize/Extra_Trees/Normal/scripts','03_experiments/model_optimization'):
    sys.path.insert(0,str(ROOT/p))
from exp006_size_com_comparison import process,PixelScaler255
from exp_cmb005_balanced_scale_stroke import prepareraw
from exp016_high_fraction_grid import nested_subset,scale_base,stroke_jitter as mlp_stroke
from exp014_balanced_augmentation import balanced_indices
from EXP004_KNN_Normal_CoM import BatchedCoM
from EXP011_KNN_Normal_Deskew import deskew,test_deskew
from EXP012_KNN_Normal_HOG import hog_features,hybrid_features,test_hog
from final_knn_normal import assemble_training
from EXP007_KNN_Normal_Augmentation import size_jitter,stroke_jitter as knn_stroke
from EXP_CMB_003_KNN_Combined_Handmade_Augmentation import balanced_nested_indices,SEED as HAND_SEED,AUG_STRENGTH
from EXP_CMB_002_KNN_Combined_Optimized import fresh_knn
from EXP_001_SVC_Normal_Baseline import baseline as svc_baseline
from EXP_CMB_005_SVC_Combined_Targeted_DB_Augmentation import TARGET_CLASSES,TARGET_SEED
from run_extra_trees import extract as et_extract
W=(41,42,43,44);SEED=42
MODELS=['MLP_Normal','MLP_Combined','SVC_Normal','SVC_Combined','KNN_Normal','KNN_Combined','HGB_Normal','HGB_Combined','ExtraTrees_Normal','ExtraTrees_Combined']
COUNTS=dict(zip(MODELS,(76800,79360,76800,82080,107520,113920,76800,79360,76800,79360)))
def digest(p):
    with open(p,'rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def write_json(p,d):Path(p).write_text(json.dumps(d,ensure_ascii=False,indent=2,default=str)+'\n',encoding='utf-8')
def environment():
    return {'python':platform.python_version(),'platform':platform.platform(),'cpu':next((s.split(':',1)[1].strip() for s in Path('/proc/cpuinfo').read_text().splitlines() if s.startswith('model name')),platform.processor()),'cpu_affinity':sorted(os.sched_getaffinity(0)),'threads':2,'versions':{p:importlib.metadata.version(p) for p in ('numpy','scipy','scikit-learn','scikit-image','joblib','opencv-python-headless')},'threadpools':threadpool_info()}
def read_training():
    audit=json.loads((BASE/'development/source_audit.json').read_text())
    assert audit['official_source_sha256']=='0e9570d0eb04c0c1da99d043b85bfdab7f3b639eebe4b10335b05b57ad671879'
    assert not audit['original_test_accessed'] and not audit['new_validation_accessed']
    with np.load(BASE/'development/official_development_only.npz',allow_pickle=False) as z:off={k:z[k] for k in z.files}
    with np.load(BASE/'development/handmade_training_only.npz',allow_pickle=False) as z:wr={w:{'X':z[f'X{w}'].astype(np.float32),'y':z[f'y{w}'].astype(np.int64),'sha256':audit['handmade_sources'][str(w)]} for w in W}
    off['sha256']=audit['official_source_sha256']
    assert off['X_train'].shape==(76800,784) and off['X_val'].shape==(19200,784)
    assert all(wr[w]['X'].shape==(640,784) and np.array_equal(np.bincount(wr[w]['y'],minlength=16),np.full(16,40)) for w in W)
    return off,wr
def hybrid(raw):
    a=BatchedCoM().transform(np.asarray(raw,dtype=np.float32).reshape(-1,784));b,_=deskew(a);del a
    h=hog_features(b);return hybrid_features(b,h,pixel_weight=.2)
def feature(name,raw):
    raw=np.asarray(raw,dtype=np.float32).reshape(-1,784)
    if name.startswith('MLP'):return process(raw,'com',20)
    if name.startswith(('KNN','SVC')):return hybrid(raw)
    return et_extract(process(raw,'com',20),'pixels+hog')
def make_model(name):
    if name.startswith('MLP'):
        h,a,b=((256,128),.03,200) if name=='MLP_Normal' else ((256,),.0001,128)
        return Pipeline([('scale_0_1',PixelScaler255()),('model',MLPClassifier(hidden_layer_sizes=h,alpha=a,batch_size=b,learning_rate_init=.001,random_state=42,activation='relu',solver='adam',early_stopping=False,max_iter=200,tol=.0001,n_iter_no_change=10))])
    if name.startswith('SVC'):
        svc,scaled,_=svc_baseline();assert scaled and svc.kernel=='rbf'
        return clone(svc).set_params(C=1.,gamma='scale',cache_size=200)
    if name.startswith('KNN'):return fresh_knn().set_params(n_jobs=2)
    if name.startswith('HGB'):return HistGradientBoostingClassifier(learning_rate=.1,max_leaf_nodes=31,l2_regularization=1.,max_iter=300,early_stopping=True,validation_fraction=.1,n_iter_no_change=10,random_state=42)
    return ExtraTreesClassifier(n_estimators=100,max_features=.18,min_samples_leaf=1,max_depth=None,random_state=42,n_jobs=2)
def hand_aug(raw,indices,seed):
    resized,_=size_jitter(raw,indices,seed=seed);altered,_=knn_stroke(resized,strength=AUG_STRENGTH);return hybrid(altered)
def official_only_30(off):
    from exp015_scale_stroke_grid import nested_subset as low_subset
    x,y=off['X_train'],off['y_train'];main=process(x,'com',20);idx,t=low_subset(y,.3);half,_=balanced_indices(y,seed=20261116)
    resized=scale_base(x,idx,t);changed,stats=mlp_stroke(resized,.28,idx,half);main[idx]=process(changed,'com',20)
    return main,{'n_aug':len(idx),**stats}
def train_features(name,off,wr):
    x=off['X_train'];y=off['y_train'];combo=name.endswith('Combined');more={}
    if name=='MLP_Normal':
        Y=np.asarray(y,dtype=np.int32);out=process(x,'com',20);idx,t=nested_subset(Y,.8);half,_=balanced_indices(Y,seed=20261116)
        resized=scale_base(x,idx,t);changed,stats=mlp_stroke(resized,.14,idx,half);out[idx]=process(changed,'com',20);assert len(idx)==61440
        return out,Y,{'augmented_rows':len(idx),**stats}
    if name in ('HGB_Normal','ExtraTrees_Normal'):
        a,meta=official_only_30(off);return et_extract(a,'pixels+hog'),y,meta
    if name.startswith(('MLP','HGB','ExtraTrees')):
        a,b,stats,_=prepareraw(off,wr,'scale30_stroke28')
        if name.startswith(('HGB','ExtraTrees')):
            a=et_extract(a,'pixels+hog');b={w:et_extract(b[w],'pixels+hog') for w in W}
        a=np.concatenate([a]+[b[w] for w in W]);y=np.concatenate([y]+[wr[w]['y'] for w in W])
        return a,np.asarray(y,dtype=np.int32),stats
    if name.startswith('KNN'):a,y=assemble_training(x,y,BatchedCoM())
    else:a=hybrid(x)
    if not combo:return a,y,more
    raw=np.concatenate([wr[w]['X'] for w in W]);ry=np.concatenate([wr[w]['y'] for w in W]);wid=np.repeat(np.asarray(W),640)
    ix=balanced_nested_indices(wid,ry,seed=HAND_SEED);allidx=ix[100];orig=hybrid(raw);aug=hand_aug(raw,allidx,HAND_SEED+100);ay=ry[allidx]
    if name.startswith('KNN'):
        q=ix[25];r=np.setdiff1d(ix[50],ix[25],assume_unique=True);assert len(q)==len(r)==len(ry)//4
        assert np.array_equal(allidx,np.arange(len(ry)))
        a=np.concatenate([a,orig,aug,orig[q],aug[r]]);y=np.concatenate([y,ry,ay,ry[q],ay[r]])
        more={'handmade_original':len(ry),'handmade_augmented':len(ry),'handmade_original_repeat':len(q),'handmade_augmented_repeat':len(r)}
    else:
        rng=np.random.default_rng(TARGET_SEED);target=[]
        for w in W:
            for c in TARGET_CLASSES:
                pool=np.flatnonzero((wid==w)&(ry==c));assert len(pool)==40;target.append(rng.permutation(pool)[:20])
        target=np.sort(np.concatenate(target));assert len(target)==160
        tg=hand_aug(raw,target,TARGET_SEED+1000);a=np.concatenate([a,orig,aug,tg]);y=np.concatenate([y,ry,ay,ry[target]])
        more={'handmade_original':len(ry),'handmade_augmented':len(ry),'handmade_targeted':len(target)}
    return a,y,more
def main(name,out):
    assert name in MODELS and sklearn.__version__=='1.8.0'
    out.mkdir(parents=True,exist_ok=False)
    with threadpool_limits(limits=2):
        off,wr=read_training();model=make_model(name)
        t=time.perf_counter();a,y,meta=train_features(name,off,wr);prep=time.perf_counter()-t
        assert a.shape==(COUNTS[name],784 if name.startswith('MLP') else 1108) and y.shape==(COUNTS[name],)
        print('FIT START',name,a.shape,flush=True);t=time.perf_counter()
        with warnings.catch_warnings(record=True) as seen:
            warnings.simplefilter('always');model.fit(a,y)
        fit=time.perf_counter()-t;del a,y;gc.collect()
        path=out/'model.joblib';probe=feature(name,off['X_train'][:16]);ref=model.predict(probe)
        joblib.dump(model,path,compress=3);del model;gc.collect();m=joblib.load(path)
        assert np.array_equal(ref,m.predict(probe)) and np.array_equal(m.classes_,np.arange(16))
        manifest={'model':name,'training_rows':COUNTS[name],'training_prepare_s':prep,'fit_s':fit,'training_time_s':prep+fit,'checkpoint_sha256':digest(path),'checkpoint_bytes':path.stat().st_size,'config':m.get_params(deep=True),'augmentation':meta,'environment':environment(),'warnings':[str(w.message) for w in seen],'validation_loaded_during_fit':False,'original_test_used':False,'professor_test_used':False}
        write_json(out/'fit_manifest.json',manifest)
        pred=m.predict(feature(name,off['X_val']));manifest['official_dev_accuracy']=float(np.mean(pred==off['y_val']));manifest['official_dev_correct']=int(np.sum(pred==off['y_val']))
        reference={'MLP_Normal':18551,'SVC_Normal':18983,'KNN_Normal':18851}
        if name in reference:manifest['reference_correct']=reference[name];manifest['reference_match']=manifest['official_dev_correct']==reference[name]
        write_json(out/'fit_manifest.json',manifest);print('FIT COMPLETE',name,manifest['training_time_s'],manifest['official_dev_accuracy'],flush=True)
if __name__=='__main__':main(sys.argv[1],Path(sys.argv[2]))


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")